In [43]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
#import anndata as ad
# anndata incompatible with tribus dependencies
# additional pyyaml, toolz, zipp installation needed for anndata from conda forge

In [ ]:
# set up paths
pdrive_mount_dir = "/run/user/1001/gvfs/smb-share:server=group3.ad.helsinki.fi,share=h30492/"

segm_cells_dir = os.path.join(pdrive_mount_dir, "farkkilab2/7_TLS/Data/Exp_4/csv")
segm_cells_qced_dir = os.path.join(pdrive_mount_dir, "farkkilab2/9_EyeMT/Data_analysis/cycif_processing/phenotyping/tribus_multiple_runs_test/input_segm_cells_qced")
pt_names = ["7-S026_iOme1_01253_4A", "7-S026_iOme2_01253_4A", "7-S046_iOme_01253_4A"]

logic_table_dir = os.path.join(pdrive_mount_dir, "farkkilab2/9_EyeMT/Data_analysis/cycif_processing/phenotyping/tribus_multiple_runs_test/tribus_logic_tables/global_immune")
logic_table_list = [os.path.join(dp, f) for dp, dn, fn in os.walk(os.path.expanduser(logic_table_dir)) for f in fn]

output_dir = os.path.join(pdrive_mount_dir, "farkkilab2/9_EyeMT/Data_analysis/cycif_processing/phenotyping/tribus_multiple_runs_test/tribus_output_global_immune_norm_filt_markers")
#segm_cells_pt_list = [os.path.join(segm_cells_dir, pt + '.csv') for pt in pt_names]

# set up depth param
depth = 2

markers_list = ['PanCK', 'Vimentin', 'Iba1', 'CD11c', 'CD4', 'CD8a', 'NKG2a'] # matching logic tables
#markers_list_colnames = ['PanCK_2', 'Vimentin_1', 'Iba1_1', 'CD11c_1', 'CD4_1', 'CD8a_2', 'NKG2A_2'] # raw signal
# normalised + filtered signal
markers_list_colnames = ['PanCK_normalized', 'Vimentin_normalized', 'Iba1_filt_norm', 'CD11c_normalized', 
                         'CD4_filt_norm', 'CD8a_filt_norm', 'NKGA_filt_norm']


In [45]:
# read qced cell segmentation from anndata
segm_cells_qced_path = os.path.join(pdrive_mount_dir, "farkkilab2/7_TLS/Data/Exp_4/qc/results/qc2_all_06_08.h5ad")

segm_cells_qced = ad.read_h5ad(segm_cells_qced_path) 
print(set(segm_cells_qced.obs.Sample))

{'7-S084_iOme_01255_4A', '7-S106_iOme_01255_4A', '7-S083_iOme1_01253_4A', '7-S130_iOme_01253_4A', '7-S098_iOme_01253_4A', '7-S081_iOme_01255_4A', '7-S118_iOme_01253_4A', '7-S229_iOme_01254_4A', '7-S121_iOme_01253_4A', '7-S378_iOme_01256_4A', '7-S350_iOme_01253_4A', '7-S131_iOme_01253_4A', '7-S332_iOme_01253_4A', '7-S095_iOme1_01253_4A', '7-S026_iOme2_01253_4A', '7-S107_iOme_01253_4A', '7-S049_iOme_01253_4A', '7-S015_iOme_01255_4A', '7-S267_iOme_01253_4A', '7-S195_iOme1_01253_4A', '7-S113_iOme_01256_4A', '7-S308_iOme_01253_4A', '7-S188_iOme_01253_4A', '7-S057_iOme_01253_4A', '7-S065_iOme_01253_4A', '7-S080_iOme2_01254_4A', '7-S123_iOme_01256_4A', '7-S355_iOme_01256_4A', '7-S063_iOme_01253_4A', '7-S197_iOme_01253_4A', '7-S112_iOme_01256_4A', '7-S189_iOme_01253_4A', '7-S100_iOme_01253_4A', '7-S073_iOme1_01253_4A', '7-S247_iOme_01255_4A', '7-S120_iOme_01256_4A', '7-S380_iOme_01255_4A', '7-S222_iOme_01253_4A', '7-S225_iOme_01255_4A', '7-S311_iOme_01256_4A', '7-S091_iOme1_01256_4A', '7-S046_

In [63]:
for pt_name in pt_names:

    print(pt_name)
    
    # filter to sample
    segm_cells_pt = segm_cells_qced[segm_cells_qced.obs.Sample == pt_name]
    
    # get cell metadata
    segm_cells_pt_obs = segm_cells_pt.obs
    
    # get marker values
    segm_cells_pt_df = segm_cells_pt.to_df()
    
    # filter to important markers
    segm_cells_pt_df = segm_cells_pt_df[segm_cells_pt_df.columns.intersection(markers_list_colnames)]
    
    
    # merge marker values + cells metadata
    segm_cells_pt_all = pd.concat([segm_cells_pt_obs, segm_cells_pt_df], axis=1)
    print(segm_cells_pt_all.head())
    print(segm_cells_pt_all.shape)
    print(segm_cells_pt_all.columns)
    
    # # save
    outp_segm_qced_path = os.path.join(segm_cells_qced_dir, pt_name + '_segm_qced2.csv')
    segm_cells_pt_all.to_csv(outp_segm_qced_path, index=True)
    
    print('saved')

7-S026_iOme1_01253_4A
    CellID sample_id                 Sample    X_centroid   Y_centroid   Area  \
3       10    S026_1  7-S026_iOme1_01253_4A  11882.488722  1312.323308  133.0   
13     100    S026_1  7-S026_iOme1_01253_4A  11927.652174  1374.420290   69.0   
33    1000    S026_1  7-S026_iOme1_01253_4A  11710.592233  1603.524272  103.0   
61   10000    S026_1  7-S026_iOme1_01253_4A  13165.109489  2385.445255  137.0   
94  100000    S026_1  7-S026_iOme1_01253_4A  11294.964286  5459.857143  112.0   

    Eccentricity imageid  CD4_filt_norm  CD8a_filt_norm  Iba1_filt_norm  \
3       0.711529  S026_1       6.116547        4.615649        4.615246   
13      0.868638  S026_1       5.630154        4.338710        5.307127   
33      0.641052  S026_1       6.987315        7.406698        4.524997   
61      0.653948  S026_1       5.447801        6.494524        5.876678   
94      0.628074  S026_1       5.818998        4.369902        6.194654   

    NKGA_filt_norm  Vimentin_normalized 

In [60]:
# pt_name = pt_names[0]
# print(pt_name)

# # filter to sample
# segm_cells_pt = segm_cells_qced[segm_cells_qced.obs.Sample == pt_name]

# segm_cells_pt_df = pd.DataFrame(segm_cells_pt.X, index=segm_cells_pt.obs_names, columns=segm_cells_pt.var_names)
# segm_cells_pt_df_meta = pd.concat([segm_cells_pt.obs, segm_cells_pt_df], axis=1)

# print(segm_cells_pt_df_meta.head())
# print(segm_cells_pt_df_meta.shape)
# print(segm_cells_pt_df_meta.columns)
# print('%%%%%%%%%%%%%%%%%%%%%%%')


7-S026_iOme1_01253_4A
    CellID sample_id                 Sample    X_centroid   Y_centroid   Area  \
3       10    S026_1  7-S026_iOme1_01253_4A  11882.488722  1312.323308  133.0   
13     100    S026_1  7-S026_iOme1_01253_4A  11927.652174  1374.420290   69.0   
33    1000    S026_1  7-S026_iOme1_01253_4A  11710.592233  1603.524272  103.0   
61   10000    S026_1  7-S026_iOme1_01253_4A  13165.109489  2385.445255  137.0   
94  100000    S026_1  7-S026_iOme1_01253_4A  11294.964286  5459.857143  112.0   

    Eccentricity imageid  aSMA_filt_norm  BCL6_filt_norm  ...     HLA-DPB2  \
3       0.711529  S026_1        4.678477        7.030578  ...   117.593985   
13      0.868638  S026_1        6.062169        6.630589  ...  1067.231884   
33      0.641052  S026_1        6.538226        7.929749  ...   566.145631   
61      0.653948  S026_1        6.808277        7.483560  ...  1151.306569   
94      0.628074  S026_1        4.511744        7.135869  ...  1052.562500   

          HLA-A       

In [42]:
for pt_name in pt_names:
    for logic_table_path in logic_table_list:
        # raw signal csv
        # segm_cells_pt_path = os.path.join(segm_cells_dir, pt_name + '.csv')
        # filtered from norm+filt h5ad object
        segm_cells_pt_path = os.path.join(segm_cells_qced_dir, pt_name + '_segm_qced.csv')
        output_name = pt_name + Path(logic_table_path).stem
        
        #print(segm_cells_pt_path)
        #print(logic_table_path)
        print(output_name)
        
        #load cell csv
        sample_data = pd.read_csv(segm_cells_pt_path)
        sample_data_selected = sample_data.loc[:, 
        ['CellID', 'Y_centroid', 'X_centroid', 'Area', 'Eccentricity'] + markers_list_colnames]
        # change colnames to match logic
        sample_data_selected.columns = ['CellID', 'Y_centroid', 'X_centroid', 'Area', 'Eccentricity'] + markers_list
        
        print("cell csv loaded")
        #load logic xls
        #logic1_name = re.findall(re.compile(r'global_[a-z]*'), Path(logic_table_path).stem)[0]
        #logic2_name = re.findall(re.compile(r'immune_.*'), Path(logic_table_path).stem)[0]
        
        #logic1 = pd.read_excel(logic_table_path, sheet_name= logic1_name)
        #logic2 = pd.read_excel(logic_table_path, sheet_name= logic2_name)
        df = pd.ExcelFile(logic_table_path)
        logic = {"Global": pd.read_excel(df, df.sheet_names[0], index_col=0), "Immune": pd.read_excel(df, df.sheet_names[1], index_col=0)}

        print("logic table loaded")

        # set random seed to ensure reproducing
        labels, scores = tribus.run_tribus(np.arcsinh(sample_data_selected/5.0), logic, depth=depth, normalization=z_score, 
                                    tuning=0, sigma=1, learning_rate=1, 
                                    clustering_threshold=100, undefined_threshold=0.0005, other_threshold=0.4, random_state=42)
        
        
        result_data = sample_data_selected.join(labels)
        labels_new = labels.join(result_data["CellID"])
        labels_new.to_csv(os.path.join(output_dir, output_name + '_tribus_annotation.csv'))
        result_data.to_csv(os.path.join(output_dir, output_name + '_raw_tribus_annotated.csv'))
        
        # labels.to_csv(os.path.join(output_dir, output_name + '_tribus_annotation.csv'))
        # scores.to_csv(os.path.join(output_dir, output_name + '_raw_tribus_annotated.csv'))
        
        print('output saved')
        print('%%%%')
        

7-S026_iOme1_01253_4Aeyemt_tribus_global_zero_immune_macrocd4_neg


ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

<bound method NDFrame.head of         CellID    Y_centroid    X_centroid   Area  Eccentricity       DAPI1  \
0            1    298.184211  11367.973684  152.0      0.428027  856.960526   
1            2    960.011905  11092.892857   84.0      0.536148  329.583333   
2            3    963.448819  11175.346457  127.0      0.551244  479.299213   
3            4    995.765957  11186.542553   94.0      0.343537  201.202128   
4            5   1105.357513   9402.409326  193.0      0.489015  627.036269   
...        ...           ...           ...    ...           ...         ...   
502367  502368  15959.322222  13168.000000  180.0      0.497801  480.033333   
502368  502369  16060.954869  11646.301663  421.0      0.790180  163.277910   
502369  502370  16300.000000  10358.607143   56.0      0.570654  316.428571   
502370  502371  16681.135593  11452.672316  177.0      0.225755  306.757062   
502371  502372  16891.545455  12709.588235  187.0      0.518498  137.604278   

             BCL6_1  

<bound method NDFrame.head of         CellID    Y_centroid    X_centroid   Area  Eccentricity       DAPI1  \
0            1    298.184211  11367.973684  152.0      0.428027  856.960526   
1            2    960.011905  11092.892857   84.0      0.536148  329.583333   
2            3    963.448819  11175.346457  127.0      0.551244  479.299213   
3            4    995.765957  11186.542553   94.0      0.343537  201.202128   
4            5   1105.357513   9402.409326  193.0      0.489015  627.036269   
...        ...           ...           ...    ...           ...         ...   
502367  502368  15959.322222  13168.000000  180.0      0.497801  480.033333   
502368  502369  16060.954869  11646.301663  421.0      0.790180  163.277910   
502369  502370  16300.000000  10358.607143   56.0      0.570654  316.428571   
502370  502371  16681.135593  11452.672316  177.0      0.225755  306.757062   
502371  502372  16891.545455  12709.588235  187.0      0.518498  137.604278   

             BCL6_1  

<bound method NDFrame.head of         CellID    Y_centroid    X_centroid   Area  Eccentricity       DAPI1  \
0            1    298.184211  11367.973684  152.0      0.428027  856.960526   
1            2    960.011905  11092.892857   84.0      0.536148  329.583333   
2            3    963.448819  11175.346457  127.0      0.551244  479.299213   
3            4    995.765957  11186.542553   94.0      0.343537  201.202128   
4            5   1105.357513   9402.409326  193.0      0.489015  627.036269   
...        ...           ...           ...    ...           ...         ...   
502367  502368  15959.322222  13168.000000  180.0      0.497801  480.033333   
502368  502369  16060.954869  11646.301663  421.0      0.790180  163.277910   
502369  502370  16300.000000  10358.607143   56.0      0.570654  316.428571   
502370  502371  16681.135593  11452.672316  177.0      0.225755  306.757062   
502371  502372  16891.545455  12709.588235  187.0      0.518498  137.604278   

             BCL6_1  

...

<bound method NDFrame.head of         CellID    Y_centroid    X_centroid   Area  Eccentricity       DAPI1  \
0            1     91.275000   9806.333333  240.0      0.534751  106.650000   
1            2    203.522523  16484.189189  111.0      0.721265  997.099099   
2            3    205.961538  14517.153846   78.0      0.591836  235.564103   
3            4    393.000000   9740.058824  136.0      0.292608  139.779412   
4            5    488.654545  14704.263636  110.0      0.568422  221.736364   
...        ...           ...           ...    ...           ...         ...   
294495  294496  11456.333333   7045.509804   51.0      0.612110  211.137255   
294496  294497  11465.401316   6625.078947  152.0      0.601005  386.769737   
294497  294498  11484.874074   7250.088889  135.0      0.317883  405.266667   
294498  294499  11614.616162   7719.484848   99.0      0.492629  413.919192   
294499  294500  12117.000000   4993.000000   47.0      0.610234  172.914894   

              BCL6_1 

<bound method NDFrame.head of         CellID    Y_centroid    X_centroid   Area  Eccentricity       DAPI1  \
0            1     91.275000   9806.333333  240.0      0.534751  106.650000   
1            2    203.522523  16484.189189  111.0      0.721265  997.099099   
2            3    205.961538  14517.153846   78.0      0.591836  235.564103   
3            4    393.000000   9740.058824  136.0      0.292608  139.779412   
4            5    488.654545  14704.263636  110.0      0.568422  221.736364   
...        ...           ...           ...    ...           ...         ...   
294495  294496  11456.333333   7045.509804   51.0      0.612110  211.137255   
294496  294497  11465.401316   6625.078947  152.0      0.601005  386.769737   
294497  294498  11484.874074   7250.088889  135.0      0.317883  405.266667   
294498  294499  11614.616162   7719.484848   99.0      0.492629  413.919192   
294499  294500  12117.000000   4993.000000   47.0      0.610234  172.914894   

              BCL6_1 

<bound method NDFrame.head of         CellID    Y_centroid    X_centroid   Area  Eccentricity       DAPI1  \
0            1     91.275000   9806.333333  240.0      0.534751  106.650000   
1            2    203.522523  16484.189189  111.0      0.721265  997.099099   
2            3    205.961538  14517.153846   78.0      0.591836  235.564103   
3            4    393.000000   9740.058824  136.0      0.292608  139.779412   
4            5    488.654545  14704.263636  110.0      0.568422  221.736364   
...        ...           ...           ...    ...           ...         ...   
294495  294496  11456.333333   7045.509804   51.0      0.612110  211.137255   
294496  294497  11465.401316   6625.078947  152.0      0.601005  386.769737   
294497  294498  11484.874074   7250.088889  135.0      0.317883  405.266667   
294498  294499  11614.616162   7719.484848   99.0      0.492629  413.919192   
294499  294500  12117.000000   4993.000000   47.0      0.610234  172.914894   

              BCL6_1 

<bound method NDFrame.head of         CellID    Y_centroid    X_centroid   Area  Eccentricity       DAPI1  \
0            1     91.275000   9806.333333  240.0      0.534751  106.650000   
1            2    203.522523  16484.189189  111.0      0.721265  997.099099   
2            3    205.961538  14517.153846   78.0      0.591836  235.564103   
3            4    393.000000   9740.058824  136.0      0.292608  139.779412   
4            5    488.654545  14704.263636  110.0      0.568422  221.736364   
...        ...           ...           ...    ...           ...         ...   
294495  294496  11456.333333   7045.509804   51.0      0.612110  211.137255   
294496  294497  11465.401316   6625.078947  152.0      0.601005  386.769737   
294497  294498  11484.874074   7250.088889  135.0      0.317883  405.266667   
294498  294499  11614.616162   7719.484848   99.0      0.492629  413.919192   
294499  294500  12117.000000   4993.000000   47.0      0.610234  172.914894   

              BCL6_1 

KeyboardInterrupt: 